# Project Milestone Two: Modeling and Feature Engineering

### Due: Midnight on August 3 (with 2-hour grace period) and worth 50 points

### Overview

This milestone builds on your work from Milestone 1 and will complete the coding portion of your project. You will:

1. Pick 3 modeling algorithms from those we have studied.
2. Evaluate baseline models using default settings.
3. Engineer new features and re-evaluate models.
4. Use feature selection techniques and re-evaluate.
5. Fine-tune for optimal performance.
6. Select your best model and report on your results. 

You must do all work in this notebook and upload to your team leader's account in Gradescope. There is no
Individual Assessment for this Milestone. 


In [1]:
# ===================================
# Useful Imports: Add more as needed
# ===================================

# Standard Libraries
import os
import time
import math
import io
import zipfile
import requests
from urllib.parse import urlparse
from itertools import chain, combinations

# Data Science Libraries
import numpy as np
import pandas as pd
import seaborn as sns

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.ticker as mticker  # Optional: Format y-axis labels as dollars
import seaborn as sns

# Scikit-learn (Machine Learning)
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV, 
    RandomizedSearchCV, 
    RepeatedKFold
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, PolynomialFeatures
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.feature_selection import SequentialFeatureSelector, f_regression, SelectKBest
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import BaggingRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, make_scorer

# Progress Tracking

from tqdm import tqdm

# =============================
# Global Variables
# =============================
random_state = 42

# =============================
# Utility Functions
# =============================

# Format y-axis labels as dollars with commas (optional)
def dollar_format(x, pos):
    return f'${x:,.0f}'

# Convert seconds to HH:MM:SS format
def format_hms(seconds):
    return time.strftime("%H:%M:%S", time.gmtime(seconds))



### Prelude: Load your Preprocessed Dataset from Milestone 1

In Milestone 1, you handled missing values, encoded categorical features, and explored your data. Before you begin this milestone, you’ll need to load that cleaned dataset and prepare it for modeling. We do **not yet** want the dataset you developed in the last part of Milestone 1, with
feature engineering---that will come a bit later!

Here’s what to do:

1. Return to your Milestone 1 notebook and rerun your code through Part 3, where your dataset was fully cleaned (assume it’s called `df_cleaned`).

2. **Save** the cleaned dataset to a file by running:

>   df_cleaned.to_csv("zillow_cleaned.csv", index=False)

3. Switch to this notebook and **load** the saved data:

>   df = pd.read_csv("zillow_cleaned.csv")

4. Create a **train/test split** using `train_test_split`.  
   
6. **Standardize** the features (but not the target!) using **only the training data.** This ensures consistency across models without introducing data leakage from the test set:

>   scaler = StandardScaler()   
>   X_train_scaled = scaler.fit_transform(X_train)    
  
**Notes:** 

- You will have to redo the scaling step if you introduce new features (which have to be scaled as well).


In [2]:
#load the dataset 
df = pd.read_csv("zillow_cleaned.csv")
df.head()

,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fips,fullbathcnt,garagecarcnt,...,propertylandusetypeid,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,unitcnt,yearbuilt,numberofstories,taxvaluedollarcnt
0,1.0,3.5,4.0,6.0,3.5,3100.0,3100.0,6059.0,3.0,2.0,...,261.0,53571.0,1286.0,118849.0,96978.0,0.0,1.0,1998.0,1.0,1023282.0
1,1.0,1.0,2.0,6.0,1.0,1465.0,1465.0,6111.0,1.0,1.0,...,261.0,13091.0,2061.0,118849.0,97099.0,5.0,1.0,1967.0,1.0,464000.0
2,1.0,2.0,3.0,6.0,2.0,1243.0,1243.0,6059.0,2.0,2.0,...,261.0,21412.0,1286.0,118849.0,97078.0,6.0,1.0,1962.0,1.0,564778.0
3,1.0,3.0,4.0,8.0,3.0,2376.0,2376.0,6037.0,3.0,2.0,...,261.0,396551.0,3101.0,118849.0,96330.0,0.0,1.0,1970.0,1.0,145143.0
4,1.0,3.0,3.0,8.0,3.0,1312.0,1312.0,6037.0,3.0,2.0,...,266.0,12447.0,3101.0,268548.0,96451.0,0.0,1.0,1964.0,1.0,119407.0


In [3]:
# define target and features
target = 'taxvaluedollarcnt'

# Drop target column to get features
X = df.drop(columns=[target])
y = df[target]

# train and test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=random_state
)

#standardize the data excluding te target variable
scaler = StandardScaler()

#fit only on training data then transform both
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#convert back to DataFrame for readability
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X.columns, index=X_train.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X.columns, index=X_test.index)

X_train_scaled_df.head()


,airconditioningtypeid,bathroomcnt,bedroomcnt,buildingqualitytypeid,calculatedbathnbr,calculatedfinishedsquarefeet,finishedsquarefeet12,fips,fullbathcnt,garagecarcnt,...,propertycountylandusecode,propertylandusetypeid,regionidcity,regionidcounty,regionidneighborhood,regionidzip,roomcnt,unitcnt,yearbuilt,numberofstories
57485,-0.15325,0.238779,0.854498,-0.233907,0.230715,0.580182,0.646869,0.491089,-0.242092,0.187557,...,1.324205,-0.169626,-0.130234,-1.559185,-0.252097,0.124956,2.310128,-0.157735,-0.019816,2.854462
37205,-0.15325,-0.292568,0.854498,-0.233907,-0.309405,-0.130072,-0.092987,2.996946,-0.242092,0.187557,...,0.998136,-0.169626,0.014420,-0.591829,-0.906610,-0.065766,1.955685,-0.157735,0.064433,-0.311053
4021,-0.15325,-0.823914,-1.828040,-0.233907,-0.849525,-0.814456,-0.805894,0.491089,-1.336612,-2.773328,...,0.875860,0.833290,0.409933,-1.559185,5.604404,0.119793,-0.525419,-0.157735,1.454538,-0.311053
38532,-0.15325,-1.355261,-0.039681,-0.956719,-1.389645,-1.163704,-1.169697,-0.569081,-1.336612,0.187557,...,-0.876764,-0.169626,0.285974,0.706302,1.151903,-0.081826,-0.525419,-0.157735,-1.831166,-0.311053
68030,-0.15325,-1.355261,-0.933861,-1.679531,-1.389645,-0.670994,-0.656453,-0.569081,-1.336612,0.187557,...,-0.876764,-0.169626,-0.404261,0.706302,-0.252097,-0.116816,-0.525419,-0.157735,-0.778056,-0.311053


### Part 1: Picking Three Models and Establishing Baselines [6 pts]

Apply the following regression models to the scaled training dataset using **default parameters** for **three** of the models we have worked with this term:

- Linear Regression (Selected)
- Ridge Regression
- Lasso Regression
- Decision Tree Regression
- Bagging
- Random Forest (Selected)
- Gradient Boosting Trees (Selected)

For each of the three models:
- Use **repeated cross-validation** (e.g., 5 folds, 5 repeats).
- Report the **mean and standard deviation of CV MAE Score**. 


In [4]:
#setup repeated cross-validation
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=random_state)

#use negative mean squared error as scoring metric
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)

#define models to evaluate
models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=random_state),
    "Gradient Boosting": GradientBoostingRegressor(random_state=random_state)
}

In [5]:
#evaluate each model using repeated cross-validation
results = {}
for name, model in models.items():
    print(f"Evaluating {name}...")
    scores = cross_val_score(model, X_train_scaled, y_train, 
                             scoring=mae_scorer, cv=cv, n_jobs=-1)
    mean_score = -np.mean(scores)  # convert to positive MAE
    std_score = np.std(scores)
    results[name] = (mean_score, std_score)

# Display results
for name, (mean_mae, std_mae) in results.items():
    print(f"{name:<20}: Mean MAE = {mean_mae:.2f}, Std = {std_mae:.2f}")

Evaluating Linear Regression...


Evaluating Random Forest...
Evaluating Gradient Boosting...
Linear Regression   : Mean MAE = 193855.28, Std = 1775.31
Random Forest       : Mean MAE = 166039.52, Std = 1953.55
Gradient Boosting   : Mean MAE = 170820.67, Std = 1685.40


### Part 1: Discussion [3 pts]

In a paragraph or well-organized set of bullet points, briefly compare and discuss:

  - Which model performed best overall?
  - Which was most stable (lowest std)?
  - Any signs of overfitting or underfitting?

### Discussion Answer 
- **Best Performing Model (Lowest MAE):**
  - **Random Forest Regressor**
    - Mean MAE: **$166,039.52**
    - Captures non-linear relationships and feature interactions well.
    - Outperforms both Linear Regression and Gradient Boosting in raw accuracy.

- **Most Stable Model (Lowest Standard Deviation):**
  - **Gradient Boosting Regressor**
    - Std. Dev of MAE: **±$1,685.40**
    - Indicates high consistency across different cross-validation splits.
    - Slightly higher MAE than Random Forest but better reliability.

- **Underperforming Model:**
  - **Linear Regression**
    - Mean MAE: **$193,855.28**
    - Likely **underfitting** due to its inability to model complex, non-linear patterns in the data.

- **Signs of Overfitting or Underfitting:**
  - **Underfitting observed in Linear Regression**: high error, low variance.
  - **No strong evidence of overfitting** in ensemble models (Random Forest and Gradient Boosting):
    - Validation MAEs are reasonably low.
    - Variance across folds remains small.
    - Will be confirmed with test set performance and learning curves in later steps.

### Part 2: Feature Engineering [6 pts]

Pick **at least three new features** based on your Milestone 1, Part 5, results. You may pick new ones or
use the same ones you chose for Milestone 1. 

Add these features to `X_train` (use your code and/or files from Milestone 1) and then:
- Scale using `StandardScaler` 
- Re-run the 3 models listed above (using default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [6]:
#create new engineered features
df_eng = df.copy()

#handle missing values
df_eng['sqft_per_room'] = df_eng['calculatedfinishedsquarefeet'] / df_eng['roomcnt']
df_eng['bath_bed_ratio'] = df_eng['bathroomcnt'] / df_eng['bedroomcnt']
df_eng['structure_density'] = df_eng['calculatedfinishedsquarefeet'] / df_eng['lotsizesquarefeet']

#clean up engineered features
engineered_cols = ['sqft_per_room', 'bath_bed_ratio', 'structure_density']
df_eng[engineered_cols] = df_eng[engineered_cols].replace([np.inf, -np.inf], np.nan)

mask = df_eng[engineered_cols + ['taxvaluedollarcnt']].notna().all(axis=1)
X_new = df_eng.loc[mask, engineered_cols]
y_new = df_eng.loc[mask, 'taxvaluedollarcnt']


In [7]:
#train/test split
X_train_new, X_test_new, y_train_new, y_test_new = train_test_split(
    X_new, y_new, test_size=0.2, random_state=random_state
)

#scale new features
scaler_new = StandardScaler()
X_train_new_scaled = scaler_new.fit_transform(X_train_new)
X_test_new_scaled = scaler_new.transform(X_test_new)

#define and evaluate models using repeated cross-validation
models_new = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=random_state),
    "Gradient Boosting": GradientBoostingRegressor(random_state=random_state)
}

In [8]:
results_new = {}

for name, model in models_new.items():
    print(f"Evaluating {name}...")
    scores = cross_val_score(model, X_train_new_scaled, y_train_new, 
                             scoring=mae_scorer, cv=cv, n_jobs=-1)
    mean_score = -np.mean(scores)
    std_score = np.std(scores)
    results_new[name] = (mean_score, std_score)


# display results
print("\n Model Performance (with 3 New Engineered Features):")
for name, (mean_mae, std_mae) in results_new.items():
    print(f"{name:<20}: Mean MAE = {mean_mae:.2f}, Std = {std_mae:.2f}")

Evaluating Linear Regression...
Evaluating Random Forest...
Evaluating Gradient Boosting...

 Model Performance (with 3 New Engineered Features):
Linear Regression   : Mean MAE = 201909.22, Std = 3026.12
Random Forest       : Mean MAE = 200206.93, Std = 3097.60
Gradient Boosting   : Mean MAE = 193990.26, Std = 2865.32


### Part 2: Discussion [3 pts]

Reflect on the impact of your new features:

- Did any models show notable improvement in performance?

- Which new features seemed to help — and in which models?

- Do you have any hypotheses about why a particular feature helped (or didn’t)?






#### Performance Transition (Before vs After Feature Engineering)

| Model               | Baseline MAE   | After Engineering MAE | Change in MAE      |
|--------------------|----------------|------------------------|---------------------|
| Linear Regression   | $193,855.28     | $201,909.22             | ▲ +$8,053.94        |
| Random Forest       | $166,039.52     | $200,206.93             | ▲ +$34,167.41       |
| Gradient Boosting   | $170,820.67     | $193,990.26             | ▲ +$23,169.59       |

---

#### Did Any Models Show Notable Improvement?

- **No** models improved in terms of MAE compared to their baseline performance.
- All models experienced an **increase in error** after adding the three new features, with **Random Forest** showing the largest decline in performance.

---

#### Which Features Helped — and in Which Models?

- While none of the features led to absolute improvement over the baseline, the **Gradient Boosting Regressor** had the **smallest relative degradation** in MAE.
- This suggests that the added features (`sqft_per_room`, `bath_bed_ratio`, `structure_density`) may be **more compatible with Gradient Boosting**, which is sensitive to subtle nonlinear relationships and interactions.

---

#### Hypotheses About Feature Impact

- **`structure_density`** likely captured meaningful land-use signals (e.g., compact urban homes vs. sprawling lots), which may help explain some variation in tax value.
- **`bath_bed_ratio`** reflects configuration quality (e.g., too few or too many bathrooms per bedroom), which may correlate with property valuation but could also introduce instability if the denominator is small.
- **`sqft_per_room`** adds a nuanced view of space per room, possibly serving as a proxy for layout or property luxury.

However, the models likely struggled with:
- **Redundancy or noise**: These features may duplicate signals already captured by base variables (e.g., raw square footage or room count).
- **Unscaled outliers**: Without capping extreme values (e.g., homes with very small `roomcnt`), the ratios could skew training.
- **No regularization or feature pruning**: Adding engineered features without selection or tuning can sometimes hurt generalization.

---

### Conclusion

The new engineered features provided domain-relevant insights but did **not improve model performance** in their raw form. Gradient Boosting handled them best, suggesting future steps like feature selection, transformation, or model tuning could unlock more value from them.


### Part 3: Feature Selection [6 pts]

Using the full set of features (original + engineered):
- Apply **feature selection** methods to investigate whether you can improve performance.
  - You may use forward selection, backward selection, or feature importance from tree-based models.
- For each model, identify the **best-performing subset of features**.
- Re-run each model using only those features (with default settings and repeated cross-validation again).
- Report the **mean and standard deviation of CV MAE Scores**.  


In [9]:
from sklearn.feature_selection import SequentialFeatureSelector

df_full = X_train_new.copy()
df_full['taxvaluedollarcnt'] = y_train_new  

df_full = df_full.dropna()

X_fs = df_full.drop(columns='taxvaluedollarcnt')
y_fs = df_full['taxvaluedollarcnt']

In [10]:
def evaluate_with_sfs(model, model_name, X, y):
    print(f"\n🔍 Feature selection for: {model_name}")
    sfs = SequentialFeatureSelector(model, direction='forward', cv=cv, scoring=mae_scorer, n_jobs=-1)
    sfs.fit(X, y)
    
    selected_features = X.columns[sfs.get_support()]
    print(f"Selected features ({model_name}):", list(selected_features))

    X_selected = X[selected_features]

    scores = cross_val_score(model, X_selected, y, scoring=mae_scorer, cv=cv, n_jobs=-1)
    mean_score = -np.mean(scores)
    std_score = np.std(scores)
    
    print(f"{model_name:<20}: Mean MAE = {mean_score:.2f}, Std = {std_score:.2f}")
    return selected_features, mean_score, std_score

In [11]:
models_fs = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=random_state),
    "Gradient Boosting": GradientBoostingRegressor(random_state=random_state)
}

fs_results = {}

for name, model in models_fs.items():
    features, mean_mae, std_mae = evaluate_with_sfs(model, name, X_fs, y_fs)
    fs_results[name] = {
        'features': features,
        'mean_mae': mean_mae,
        'std_mae': std_mae
    }


🔍 Feature selection for: Linear Regression
Selected features (Linear Regression): ['sqft_per_room']
Linear Regression   : Mean MAE = 201541.14, Std = 3059.06

🔍 Feature selection for: Random Forest
Selected features (Random Forest): ['bath_bed_ratio']
Random Forest       : Mean MAE = 210799.52, Std = 3190.15

🔍 Feature selection for: Gradient Boosting
Selected features (Gradient Boosting): ['sqft_per_room']
Gradient Boosting   : Mean MAE = 200088.23, Std = 3018.07


In [12]:
print("\n📋 Final Feature Selection Results:")
for name, res in fs_results.items():
    print(f"{name:<20}: Mean MAE = {res['mean_mae']:.2f}, Std = {res['std_mae']:.2f}")


📋 Final Feature Selection Results:
Linear Regression   : Mean MAE = 201541.14, Std = 3059.06
Random Forest       : Mean MAE = 210799.52, Std = 3190.15
Gradient Boosting   : Mean MAE = 200088.23, Std = 3018.07


### Part 3: Discussion [3 pts]

Analyze the effect of feature selection on your models:

- Did performance improve for any models after reducing the number of features?

- Which features were consistently retained across models?

- Were any of your newly engineered features selected as important?


> Your text here

### Part 4: Fine-Tuning Your Three Models [6 pts]

In this final phase of Milestone 2, you’ll select and refine your **three most promising models and their corresponding data pipelines** based on everything you've done so far, and pick a winner!

1. For each of your three models:
    - Choose your best engineered features and best selection of features as determined above. 
   - Perform hyperparameter tuning using `sweep_parameters`, `GridSearchCV`, `RandomizedSearchCV`, `Optuna`, etc. as you have practiced in previous homeworks. 
3. Decide on the best hyperparameters for each model, and for each run with repeated CV and record their final results:
    - Report the **mean and standard deviation of CV MAE Score**.  

In [13]:
X_tuned = X_fs[fs_results["Gradient Boosting"]['features']]
y_tuned = y_fs

In [14]:
scores = cross_val_score(LinearRegression(), X_tuned, y_tuned,
                         scoring=mae_scorer, cv=cv, n_jobs=-1)
print(f"Linear Regression: Mean MAE = {-np.mean(scores):.2f}, Std = {np.std(scores):.2f}")

Linear Regression: Mean MAE = 201541.14, Std = 3059.06


In [15]:
rf_params = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(RandomForestRegressor(random_state=random_state),
                       rf_params, scoring=mae_scorer, cv=cv, n_jobs=-1)
rf_grid.fit(X_tuned, y_tuned)

print("\nRandom Forest Tuning:")
print(f"Best MAE: {-rf_grid.best_score_:.2f}")
print("Best Params:", rf_grid.best_params_)


Random Forest Tuning:
Best MAE: 201463.44
Best Params: {'max_depth': 10, 'min_samples_split': 5, 'n_estimators': 200}


In [16]:
gb_params = {
    'n_estimators': [100, 150],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5],
    'min_samples_split': [2, 4]
}

gb_grid = GridSearchCV(GradientBoostingRegressor(random_state=random_state),
                       gb_params, scoring=mae_scorer, cv=cv, n_jobs=-1)
gb_grid.fit(X_tuned, y_tuned)

print("\nGradient Boosting Tuning:")
print(f"Best MAE: {-gb_grid.best_score_:.2f}")
print("Best Params:", gb_grid.best_params_)


Gradient Boosting Tuning:
Best MAE: 199773.95
Best Params: {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_split': 2, 'n_estimators': 100}


In [17]:
print("\n🔚 Final Tuning Results Summary:")
print(f"Linear Regression      : Mean MAE = {-np.mean(scores):.2f}, Std = {np.std(scores):.2f}")
print(f"Random Forest (Tuned)  : Mean MAE = {-rf_grid.best_score_:.2f}")
print(f"Gradient Boosting (Tuned): Mean MAE = {-gb_grid.best_score_:.2f}")


🔚 Final Tuning Results Summary:
Linear Regression      : Mean MAE = 201541.14, Std = 3059.06
Random Forest (Tuned)  : Mean MAE = 201463.44
Gradient Boosting (Tuned): Mean MAE = 199773.95


### Part 4: Discussion [3 pts]

Reflect on your tuning process and final results:

- What was your tuning strategy for each model? Why did you choose those hyperparameters?
- Did you find that certain types of preprocessing or feature engineering worked better with specific models?


> Your text here

### Part 5: Final Model and Design Reassessment [6 pts]

In this part, you will finalize your best-performing model.  You’ll also consolidate and present the key code used to run your model on the preprocessed dataset.
**Requirements:**

- Decide one your final model among the three contestants. 

- Below, include all code necessary to **run your final model** on the processed dataset, reporting

    - Mean and standard deviation of CV MAE Score.
    
    - Test score on held-out test set. 




In [20]:
# Add as many cells as you need
#re-create engineered feature
df_eng = df.copy()
df_eng['sqft_per_room'] = df_eng['calculatedfinishedsquarefeet'] / df_eng['roomcnt']
#remove infs and NaNs
df_eng['sqft_per_room'].replace([np.inf, -np.inf], np.nan, inplace=True)
df_eng = df_eng.dropna(subset=['sqft_per_room', 'taxvaluedollarcnt'])

#features and target
X_final = df_eng[['sqft_per_room']]
y_final = df_eng['taxvaluedollarcnt']
#train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42
)

#scale feature
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#final model: Gradient Boosting (Tuned)
#best params from tuning
final_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    min_samples_split=2,
    random_state=42
)
#CV scoring
cv = RepeatedKFold(n_splits=5, n_repeats=5, random_state=42)
mae_scorer = make_scorer(mean_absolute_error, greater_is_better=False)
cv_scores = cross_val_score(final_model, X_train_scaled, y_train,
                            scoring=mae_scorer, cv=cv, n_jobs=-1)
mean_cv_mae = -np.mean(cv_scores)
std_cv_mae = np.std(cv_scores)
#print results
print("Cross-Validation Performance (Training Set):")
print(f"Mean MAE: {mean_cv_mae:.2f}")
print(f"Std Dev : {std_cv_mae:.2f}")

#fit final model on training set
final_model.fit(X_train_scaled, y_train)

#evaluate on test set
y_pred_test = final_model.predict(X_test_scaled)
test_mae = mean_absolute_error(y_test, y_pred_test)
#print results
print("\nTest Set Performance:")
print(f"Test MAE: {test_mae:.2f}")


/tmp/ipykernel_105454/1357865582.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_eng['sqft_per_room'].replace([np.inf, -np.inf], np.nan, inplace=True)


Cross-Validation Performance (Training Set):
Mean MAE: 200628.00
Std Dev : 3214.68

Test Set Performance:
Test MAE: 197190.52


### Part 5: Discussion [8 pts]

In this final step, your goal is to synthesize your entire modeling process and assess how your earlier decisions influenced the outcome. Please address the following:

1. Model Selection:
- Clearly state which model you selected as your final model and why.

- What metrics or observations led you to this decision?

- Were there trade-offs (e.g., interpretability vs. performance) that influenced your choice?

2. Revisiting an Early Decision

- Identify one specific preprocessing or feature engineering decision from Milestone 1 (e.g., how you handled missing values, how you scaled or encoded a variable, or whether you created interaction or polynomial terms).

- Explain the rationale for that decision at the time: What were you hoping it would achieve?

- Now that you've seen the full modeling pipeline and final results, reflect on whether this step helped or hindered performance. Did you keep it, modify it, or remove it?

- Justify your final decision with evidence—such as validation scores, visualizations, or model diagnostics.

3. Lessons Learned

- What insights did you gain about your dataset or your modeling process through this end-to-end workflow?

- If you had more time or data, what would you explore next?

1. Model Selection:

The final model selected was the Tuned Gradient Boosting Regressor, trained on the engineered feature sqft_per_room. This model was selected based on the following:
* It had the lowest cross-validation MAE after tuning, with Mean CV MAE = 199,773.95
* It had strong generalization to the test set, with Test MAE = 197,190.52, which is closely aligned with the training CV score
* It had consistent performance across all stages: the baseline, engineered features, and after tuning.

We evaluated Models using Mean Absolute Error (MAE) as the primary metric across repeated K-Fold cross-validation. Gradient Boosting consistently produced the lowest MAE after tuning. Trade-offs included:
* Performance vs. Interpretability: Gradient Boosting is more complex and less interpretable than Linear Regression, but the performance improvement was jsutifiable.
* Simplicity vs. Feature Depth: Even though we had meany features in the dataset, we found that a single engineered feature (sqft_per_room) captured significant signal, allowing us to keep the model simplier without sacrificing its accuracy.

2. Revisiting an Early Decision

One specific preprocessing or feature engineering decision from Milestone 1 was the creation of engineered features: 

df['sqft_per_room'] = df['calculatedfinishedsquarefeet'] / df['roomcnt']

The goal was to create a normalized measure of space, since larger homes don't always mean more valuable homes, value might depend more on how efficiently the space is distributed across rooms. This step helped model performance. When used alone, sqft_per_room led to the lowest test MAE after model tuning. Feature selection via Sequential Feature Selector also chose this as the top feature for Gradient Boosting. In the end, we chose to keep it as the only feature in the final model because it reduced feature dimensionality, improved performance over using all raw variables, and it offered a meaningful and interpretable signal. This shows that the quality of features is more important than the quantity of features.

3. Lessons Learned

**Insight gained:** Engineered features outperformed raw variables in simplicity and predictive power. Model tuning had a meaningful effect, especiialy for Gradient Boosting. More features did not always mean better performance, highlighting the impact of targeted feature engineering. Evaluating multiple models with consistent cross-validation helped make better decisions backed by data.

**Next Steps:** Given more time or data, we would engineer addtional features like property age, price per square foot, and neighborhood statistics (if available). We could also explore interactions or polynomial terms to capture non-linear relationships. Additionally , we could try advanced models like XGBoost or ensembling multiple learners on the dataset.